In [18]:
import torch

In [19]:
torch.set_default_dtype(torch.float64)

In [20]:
m = 7
l = 9
vocab_size = 9
target = [0, 1, 2, 3, 4, 5, 8]
assert len(target) == m
target = torch.tensor(target)

In [21]:
transition_matrix = torch.zeros((l, l))

In [22]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.1 #prob
    ),
    (
        (2, 1), #row, col
        0.8 #prob
    ),
    (
        (2, 9), #row, col
        0.1 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),
]

In [23]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [24]:
token_probs = torch.zeros((l, vocab_size))

In [25]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.8 #prob
    ),
    (
        (1, 3), #row, col
        0.1
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.1 #prob
    ),
    (
        (2, 4), #row, col
        0.1 #prob
    ),
    (
        (2, 6), #row, col
        0.1 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    )
]

In [26]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [27]:
batch_size = 2
# repeat transition, emission, and targets to mimic batch size
transition_matrix = transition_matrix.repeat(batch_size, 1, 1)
token_probs = token_probs.repeat(batch_size, 1, 1)
target = target.repeat(batch_size, 1)

In [28]:
dp = torch.zeros((batch_size, m, l))

In [29]:
dp[:, 0, 0] = 0.9

In [12]:
transition_matrix

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8000, 0.0000, 0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [36]:
dp[:, 1 - 1, :].unsqueeze(1).shape

torch.Size([2, 1, 9])

In [41]:
dp[:, 1 - 1, :].unsqueeze(1).transpose(1,2).shape

torch.Size([2, 9, 1])

In [43]:
(dp[:, 1 - 1, :].unsqueeze(1) @ transition_matrix).shape

torch.Size([2, 1, 9])

In [49]:
tmp = (dp[:, 1 - 1, :].unsqueeze(1) @ transition_matrix)

In [50]:
tmp0 = token_probs[:, target[1]]

In [57]:
(tmp0.transpose(1,2) + tmp.transpose(1,2)).shape

torch.Size([2, 9, 7])

In [55]:
tmp.squeeze(1).shape, tmp0.shape

(torch.Size([2, 9]), torch.Size([2, 7, 9]))

In [44]:
transition_matrix.shape

torch.Size([2, 9, 9])

In [45]:
token_probs[:, target[1]].shape

torch.Size([2, 7, 9])

In [48]:
(token_probs[:, target[1]] + (dp[:, 1 - 1, :].unsqueeze(1) @ transition_matrix)).shape

torch.Size([2, 7, 9])

In [33]:
token_probs[:, target[1]] * (dp[:, 1 - 1, :] @ transition_matrix)

RuntimeError: The size of tensor a (7) must match the size of tensor b (2) at non-singleton dimension 1

In [31]:
for i in range(1, m):
    dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

RuntimeError: expand(torch.DoubleTensor{[2, 7, 9]}, size=[7, 9]): the number of sizes provided (2) must be greater or equal to the number of dimensions in the tensor (3)

In [45]:
# example log sum exp
testing_logs = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9]).float()
test_logs = torch.log_softmax(testing_logs, dim=0)

In [53]:
test_mask = torch.tensor([0, 0, 0, 0, 0, 0, 0, 0, 1]).bool()

In [54]:
test_logs

tensor([-8.4586, -7.4586, -6.4586, -5.4586, -4.4586, -3.4586, -2.4586, -1.4586,
        -0.4586], dtype=torch.float32)

In [49]:
unmasked_sum = torch.logsumexp(test_logs, dim=0)

In [55]:
test_logs_masked = test_logs.masked_fill(test_mask, float('-inf'))

In [56]:
masked_sum = torch.logsumexp(test_logs_masked, dim=0)

In [57]:
unmasked_sum, masked_sum

(tensor(0., dtype=torch.float32), tensor(-1.0002, dtype=torch.float32))

In [58]:
torch.exp(unmasked_sum), torch.exp(masked_sum)

(tensor(1., dtype=torch.float32), tensor(0.3678, dtype=torch.float32))

In [59]:
test_dp = torch.randn((2, 3, 4))

In [69]:
test_dp

tensor([[[ 1.2520,  0.0792,  0.8453, -0.1488],
         [ 0.0000, -1.2683, -1.4902,  1.9630],
         [ 0.0000,  0.0000,  0.5637, -0.0982]],

        [[ 1.1918, -0.8417, -0.6407,  0.7817],
         [ 0.0000, -0.8097, -0.2169, -1.5191],
         [ 0.0000,  0.0000,  0.1614,  1.0816]]])

In [68]:
test_dp[:, 2, :2] = 0

In [71]:
dp[-1][-1]

tensor(0.0026)

In [28]:
print(dp.round(decimals=3))

tensor([[0.9000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.1890, 0.1260, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0150, 0.0000, 0.0150, 0.0130, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0140, 0.0010, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0120, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0040, 0.0010, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0030]])


In [15]:
# doing it by hand by enumerating all possible paths
p1m = torch.tensor([0.8, 0.7, 0.8, 0.9, 0.9, 0.6, 0.7])
p1t = torch.tensor([0.3, 0.1, 1, 1, 0.5, 1])

# p2m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.1, 0.2])
# p2t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

p3m = torch.tensor([0.8, 0.2, 0.1, 0.1, 0.1, 0.2, 0.7])
p3t = torch.tensor([0.7, 1, 1, 0.5, 1, 1])

In [16]:
p1 = p1m.prod() * p1t.prod()
# p2 = p2m.prod() * p2t.prod()
p3 = p3m.prod() * p3t.prod()

In [17]:
# acc = p1 + p2 + p3
acc = p1 + p3

In [18]:
acc

tensor(0.0023)

In [70]:
torch.exp(dp[m-1, l-1])

tensor(1.0026)

In [19]:
# check difference between dp[m-1, l-1] and acc
dp[m-1, l-1] - acc

tensor(0.0003)

I understand the problem now! Before, the dynamic programming approach and the brute-force considering all paths approach were slightly different, and it was bothering me as to why, but I think I figured it out. 

It actually explains a lot too, but the point is, we want to ensure all paths end at the same vertex, in this case, vertex 9. Since the target has length 7, this means we want to consider all paths of length 7 that end up at vertex 9 (that is, all paths that contain exactly 6 edges and end at vertex 9).

Before in my brute force attempt, I was considering all paths of length 7, but did not ensure that they ended at vertex 9, in the variable `p2m`, I was ending at vertex 8, and thus my result was over what it should have been.